In [ ]:
import pandas as _hex_pandas
import datetime as _hex_datetime
import json as _hex_json

In [ ]:
hex_scheduled = _hex_json.loads("false")

In [ ]:
hex_user_email = _hex_json.loads("\"example-user@example.com\"")

In [ ]:
hex_user_attributes = _hex_json.loads("{}")

In [ ]:
hex_run_context = _hex_json.loads("\"logic\"")

In [ ]:
hex_timezone = _hex_json.loads("\"UTC\"")

In [ ]:
hex_project_id = _hex_json.loads("\"01a0f9d4-f295-71ce-b94f-dbd3a78c293f\"")

In [ ]:
hex_project_name = _hex_json.loads("\"Know thy customers💰\"")

In [ ]:
hex_status = _hex_json.loads("\"\"")

In [ ]:
hex_categories = _hex_json.loads("[]")

In [ ]:
hex_color_palette = _hex_json.loads("[\"#4C78A8\",\"#F58518\",\"#E45756\",\"#72B7B2\",\"#54A24B\",\"#EECA3B\",\"#B279A2\",\"#FF9DA6\",\"#9D755D\",\"#BAB0AC\"]")

# Presentation brief

- **Audience:** Sales and marketing executives
- **Format:** Short presentation, plus an app for anyone who wants to dig deeper
- **Style:** Simple, plain language, business-focused

## Goals
1. Show that customer loyalty is worth investing in, because there's money to be made.
2. Briefly explain how loyalty was defined and what drives it.

## What this notebook looks at
- **Loyalty threshold:** how many orders make a customer "loyal"
- **First purchase:** whether what customers buy first predicts loyalty
- **Purchase sub-category:** whether what loyal customers buy differs
- **Customer segment:** whether segment matters for loyalty
- **Markets:** how loyalty differs across markets
- **Speed of return:** whether a fast second order predicts loyalty
- **Converting Regulars:** profit gained if more near-loyal customers become loyal, by market
- **First purchase by market:** whether the first-purchase effect differs by market
- **Share of business:** how much revenue loyal customers drive

## Definitions
| Tier | Orders |
|---|---|
| One-time | 1 |
| Occasional | 2–3 |
| Regular | 4–6 |
| Loyal | 7+ |

Loyalty is never driven by discounts in this analysis.

In [ ]:
import json as _hex_json

minimum_orders = _hex_json.loads("7")

In [ ]:
# import jinja2
# raw_query = """
#     SELECT * RENAME (column00 AS line_id)
#     FROM "superstore.csv"; 
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     /* A customer's order history */
#     SELECT 
#         customer_id
#         ,order_id
#         ,ship_mode
#         ,country
#         ,market
#         ,product_id
#         ,quantity
#         ,sales 
#     FROM raw 
#     WHERE customer_id = 'BS-1800'
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     /* Is an order_id unqiue to a customer_id? */
#     SELECT 
#         order_id
#         ,COUNT(DISTINCT customer_id) AS customer_count
#         ,STRING_AGG(DISTINCT customer_id, ',') AS customer_ids
#     FROM raw 
#     GROUP BY 1
#     HAVING customer_count > 1
#     ORDER BY 2 DESC;
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     /* Set primary key as market combined with customer_id */
#     SELECT
#         customer_name
#         ,market || '-' || customer_id AS customer_key
#         ,market
#         ,segment AS customer_segments
#         ,STRING_AGG(DISTINCT category, ', ' ORDER BY category) AS product_categories
#         ,COUNT(DISTINCT order_id) AS orders
#         ,ROUND(SUM(sales),2) AS sales
#         ,ROUND(SUM(profit),2) AS profit
#         ,ROUND(SUM(discount * sales) / SUM(sales),3) AS avg_discount
#     FROM raw
#     GROUP BY 1, 2, 3, 4
# """
# sql_query = jinja2.Template(raw_query).render(vars())

## What makes a customer loyal ? ## 

In [ ]:
# import jinja2
# raw_query = """
#     WITH dist AS (
#         -- How many customers have exactly N orders
#         SELECT
#             orders
#             ,COUNT(*) AS customers
#         FROM customers
#         GROUP BY orders
#     )
#     
#     -- Running total from the top = customers with N or more orders
#     SELECT
#         orders
#         ,customers
#         ,SUM(customers) OVER (ORDER BY orders DESC) AS customers_at_or_above
#         ,SUM(customers) OVER (ORDER BY orders DESC)/ SUM(customers) OVER () AS share_at_or_above
#         ,orders = {{ minimum_orders }} AS is_selected_threshold
#     FROM dist
#     ORDER BY orders
#     
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     SELECT
#         customer_segments
#         ,COUNT(*) AS customers
#         ,AVG(CASE WHEN orders >= {{ minimum_orders }} THEN 1.0 ELSE 0 END) AS loyal_rate
#         ,SUM(profit) / COUNT(*) AS profit_per_customer
#     FROM customers
#     GROUP BY 1
#     ORDER BY 1
# """
# sql_query = jinja2.Template(raw_query).render(vars())

## Distribution of loyal customers by market ##

In [ ]:
# import jinja2
# raw_query = """
#     /* Is a particular market more loyal than the others? */ 
#     SELECT
#         market
#         ,COUNT(*) AS customers
#         ,AVG(CASE WHEN orders >= {{ minimum_orders }} THEN 1.0 ELSE 0 END) AS loyal_rate
#         ,SUM(profit) / COUNT(*) AS profit_per_customer
#         ,SUM(CASE WHEN orders >= {{ minimum_orders }} THEN profit ELSE 0 END) / SUM(profit) AS profit_from_loyal_customers
#     FROM customers
#     GROUP BY 1
#     ORDER BY loyal_rate DESC
# """
# sql_query = jinja2.Template(raw_query).render(vars())

## Break down customers with more details ## 

In [ ]:
# import jinja2
# raw_query = """
#     WITH first_orders AS (
#         SELECT
#             market || '-' || customer_id AS customer_key
#             ,order_id
#         FROM raw
#         GROUP BY 1, 2
#         QUALIFY ROW_NUMBER() OVER (
#             PARTITION BY market || '-' || customer_id
#             ORDER BY MIN(order_date), order_id
#         ) = 1
#     )
#     
#     SELECT
#         c.customer_key
#         ,c.customer_name
#         ,c.market
#         ,c.customer_segments
#         ,c.orders
#         ,r.order_id
#         ,r.order_date
#         ,r.category
#         ,r.sub_category
#         ,r.product_name
#         ,r.sales
#         ,r.profit
#         ,f.order_id IS NOT NULL AS is_first_order
#     FROM raw r
#     JOIN customers c
#         ON c.customer_key = r.market || '-' || r.customer_id
#     LEFT JOIN first_orders f
#         ON f.customer_key = c.customer_key
#         AND f.order_id = r.order_id
# """
# sql_query = jinja2.Template(raw_query).render(vars())

## What creates loyalty? ##
- Acquisition behavior 
- Understand who the customers are and what they buy 
- How fast customers come back to Superstore to make their second purchase 

In [ ]:
# import jinja2
# raw_query = """
#     /* Which first purchase turns customers into loyal ones? This question focuses on what to offer new customers */
#     
#     SELECT
#         sub_category
#         ,COUNT(*) AS customers
#         ,AVG(CASE WHEN orders >= {{ minimum_orders }} THEN 1.0 ELSE 0 END) AS loyal_rate
#     FROM (
#         SELECT DISTINCT
#             customer_key
#             ,sub_category
#             ,orders
#         FROM customer_lines
#         WHERE is_first_order 
#     )
#     GROUP BY 1
#     ORDER BY loyal_rate DESC
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
# import jinja2
# raw_query = """
#     /* Loyal customers only: share of their spend by sub-category, and its margin.
#        Spend = SUM(sales); margin = SUM(profit) / SUM(sales) */
#     SELECT
#         sub_category
#         ,SUM(sales) AS loyal_sales
#         ,SUM(sales) / SUM(SUM(sales)) OVER () AS share_of_loyal_spend
#         ,SUM(profit) / SUM(sales) AS margin
#     FROM customer_lines
#     WHERE orders >= {{ minimum_orders }}
#     GROUP BY 1
#     ORDER BY loyal_sales DESC
#     
#     
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
import json as _hex_json

market = _hex_json.loads("\"All\"")

In [ ]:
# import jinja2
# raw_query = """
#     /* How quickly did each customer place their 2nd order, and did they become 'loyal'?
#        Customers with only 1 order are excluded (they never placed a 2nd) */
#        
#     WITH orders AS (
#         SELECT
#             customer_key
#             ,order_id
#             ,MIN(order_date) AS order_date
#         FROM customer_lines
#         GROUP BY 1, 2
#     ),
#     numbered AS (
#         SELECT
#             *
#             ,ROW_NUMBER() OVER (PARTITION BY customer_key ORDER BY order_date, order_id) AS order_number
#         FROM orders
#     ),
#     second_gap AS (
#         -- days from first to second order 
#         SELECT
#             first.customer_key
#             ,DATE_DIFF('day', first.order_date, second.order_date) AS days_to_second_order
#         FROM numbered first
#         JOIN numbered second
#             ON second.customer_key = first.customer_key
#             AND second.order_number = 2
#         WHERE first.order_number = 1
#     )
#     SELECT
#         CASE
#             WHEN g.days_to_second_order <= 90 THEN '1. Within 3 months'
#             WHEN g.days_to_second_order <= 180 THEN '2. 3-6 months'
#             WHEN g.days_to_second_order <= 365 THEN '3. 6-12 months'
#             ELSE '4. Over 12 months'
#         END AS time_to_second_order
#         ,COUNT(*) AS customers
#         ,AVG(CASE WHEN c.orders >= {{ minimum_orders }} THEN 1.0 ELSE 0 END) AS loyal_rate
#         ,SUM(c.profit) / COUNT(*) AS profit_per_customer
#     FROM second_gap g
#     JOIN customers c USING (customer_key)
#     WHERE ({{ market }} = 'All' OR c.market = {{ market }})
#     GROUP BY 1
#     
# """
# sql_query = jinja2.Template(raw_query).render(vars())

## How do we capture even more loyalty? ##
| Characterize Purchase Habits | # of Orders |
|---|---|
| One-time | 1 |
| Occasional | 2 to 3 |
| Regular | 4 to 6 |
| Loyal | 7 or more |

In [ ]:
import json as _hex_json

convert_pct = _hex_json.loads("10")

In [ ]:
# 1. Split customers (one row per customer, from SQL) into two groups
#    Fixed definitions: Regular = 4-6 orders, Loyal = 7+ orders
regulars = customers[customers.orders.between(4, 6)]
loyal = customers[customers.orders >= 7]

# 2. Per market: how many regulars, and average profit of each group
regular_stats = regulars.groupby("market").agg(
    regulars=("customer_key", "count"),
    regular_profit=("profit", "mean"))
loyal_stats = loyal.groupby("market").agg(
    loyal_profit=("profit", "mean"))

# 3. Put them side by side, one row per market (Canada drops: no Loyal customers)
mc = regular_stats.join(loyal_stats, how="inner").reset_index()

# 4. What-if: convert_pct% of regulars become Loyal
mc["gain_per_customer"] = mc.loyal_profit - mc.regular_profit
mc["customers_converted"] = mc.regulars * convert_pct / 100
mc["profit_gain"] = mc.customers_converted * mc.gain_per_customer
market_conversion = mc.sort_values("profit_gain", ascending=False)

# Total across markets where converting pays (Africa excluded)
total_profit_gain = market_conversion.profit_gain[market_conversion.profit_gain > 0].sum()

In [ ]:
market_conversion

In [ ]:
# import jinja2
# raw_query = """
#     /* Per market: do customers whose FIRST order included furniture
#        (Tables, Chairs, Furnishings) become loyal more often than others? */
#     WITH first_purchase AS (
#         -- A customer's first order
#         SELECT
#             customer_key
#             ,market
#             ,MAX(CASE WHEN sub_category IN ('Tables', 'Chairs', 'Furnishings') THEN 1 ELSE 0 END)
#                 AS started_with_furniture                                   -- 1 if ANY line was furniture
#             ,CASE WHEN orders >= {{ minimum_orders }} THEN 1.0 ELSE 0 END AS is_loyal
#         FROM customer_lines
#         WHERE is_first_order
#         GROUP BY customer_key, market, orders
#     )
#     SELECT
#         market
#         ,COUNT(*) AS customers
#         ,AVG(started_with_furniture) AS share_started_with_furniture
#         ,AVG(is_loyal) FILTER (WHERE started_with_furniture = 1) AS loyal_rate_furniture_first
#         ,AVG(is_loyal) FILTER (WHERE started_with_furniture = 0) AS loyal_rate_other_first
#         ,loyal_rate_furniture_first - loyal_rate_other_first AS gateway_lift -- measures who likely a customer is to become loyal if their first order includes furniture
#     FROM first_purchase
#     GROUP BY market
#     ORDER BY gateway_lift DESC
# """
# sql_query = jinja2.Template(raw_query).render(vars())

In [ ]:
import json as _hex_json

order_year = _hex_json.loads("\"All\"")

In [ ]:
# import jinja2
# raw_query = """
#     SELECT
#         CASE WHEN orders >= {{ minimum_orders }} THEN 'Loyal' ELSE 'Not loyal' END AS customer_group
#         ,COUNT(DISTINCT customer_key) AS customers
#         ,SUM(sales) AS revenue
#         ,SUM(sales) / COUNT(DISTINCT customer_key) AS revenue_per_customer
#         ,SUM(profit) AS profit
#         ,COUNT(DISTINCT customer_key) / SUM(COUNT(DISTINCT customer_key)) OVER () AS share_of_customers
#         ,SUM(sales) / SUM(SUM(sales)) OVER () AS share_of_revenue
#     FROM customer_lines
#     WHERE ({{ order_year }} = 'All' OR CAST(YEAR(order_date) AS VARCHAR) = {{ order_year }})
#     GROUP BY 1
#     ORDER BY 1
# """
# sql_query = jinja2.Template(raw_query).render(vars())